# 06-1 BART 모델로 텍스트 요약하기

<table align="left"><tr><td>
<a href="https://colab.research.google.com/github/rickiepark/hm-dl/blob/main/06-1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="코랩에서 실행하기"/></a>
</td></tr></table>

## 트랜스포머 인코더-디코더 모델 만들기

In [1]:
import keras
import keras_nlp

2026-04-08 10:11:44.741318: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
2026-04-08 10:11:44.887399: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-04-08 10:11:47.399255: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
/home/an9383/.local/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
def transformer_decoder(x, encoder_output, padding_mask, encoder_padding_mask,
                        dropout, activation='relu'):
    # 어텐션 마스크를 계산합니다.
    attention_mask = AttentionMask()(padding_mask)
    # 스킵 연결을 준비합니다.
    residual = x
    key_dim = hidden_dim // num_heads
    # 멀티 헤드 어텐션을 통과합니다.
    x = layers.MultiHeadAttention(num_heads, key_dim, dropout=dropout)(
        query=x, value=x, attention_mask=attention_mask)
    x = layers.Dropout(dropout)(x)
    # 스킵 연결
    x = x + residual
    x = layers.LayerNormalization()(x)

    # 스킵 연결을 준비합니다.
    residual = x
    # 크로스 어텐션을 통과합니다.
    x = layers.MultiHeadAttention(num_heads, key_dim, dropout=dropout)(
        query=x, value=encoder_output, attention_mask=encoder_padding_mask)
    x = layers.Dropout(dropout)(x)
    # 스킵 연결
    x = x + residual
    x = layers.LayerNormalization()(x)

    # 스킵 연결을 준비합니다.
    residual = x
    # 위치별 피드 포워드 네트워크
    x = layers.Dense(hidden_dim * 4, activation=activation)(x)
    x = layers.Dense(hidden_dim)(x)
    x = layers.Dropout(dropout)(x)
    # 스킵 연결
    x = x + residual
    x = layers.LayerNormalization()(x)
    return x

## BART 모델로 텍스트 요약하기

In [3]:
def make_causal_mask(seq_len):
    n_hori = keras.ops.arange(seq_len)
    n_vert = keras.ops.expand_dims(n_hori, axis=-1)
    mask = n_vert >= n_hori
    return mask

def make_attention_mask(padding_mask):
    # padding_mask 크기가 (2, 5)라고 가정해 보죠.
    batch_size, seq_len = keras.ops.shape(padding_mask)
    # causal_mask 크기는 (5, 5)가 됩니다.
    causal_mask = make_causal_mask(seq_len)
    # 배치 차원을 추가해 (2, 5, 5)로 만듭니다.
    causal_mask = keras.ops.broadcast_to(causal_mask, (batch_size, seq_len, seq_len))
    # 브로드캐스팅을 위해 padding_mask 크기를 (2, 1, 5)로 만듭니다.
    padding_mask = keras.ops.expand_dims(padding_mask, axis=1)
    return keras.ops.minimum(causal_mask, padding_mask)

class AttentionMask(keras.Layer):
    def call(self, padding_mask):
        return make_attention_mask(padding_mask)

In [4]:
# x는 토큰 임베딩과 위치 임베딩을 더한 값입니다.
def transformer_encoder(x, padding_mask, dropout, activation='relu'):
    residual = x
    key_dim = hidden_dim // num_heads
    # 멀티 헤드 어텐션을 통과합니다.
    x = layers.MultiHeadAttention(num_heads, key_dim, dropout=dropout)(
        query=x, value=x, attention_mask=padding_mask)
    x = layers.Dropout(dropout)(x)
    # 스킵 연결
    x = x + residual
    x = layers.LayerNormalization()(x)
    residual = x
    # 위치별 피드 포워드 네트워크
    x = layers.Dense(hidden_dim * 4, activation=activation)(x)
    x = layers.Dense(hidden_dim)(x)
    x = layers.Dropout(dropout)(x)
    # 스킵 연결
    x = x + residual
    x = layers.LayerNormalization()(x)
    return x

In [5]:
from keras import layers

In [6]:
# BART
vocab_size = 50265
num_layers = 6
num_heads = 12
hidden_dim = 768
dropout = 0.1
activation = 'gelu'
max_seq_len = 1024

encoder_token_ids = keras.Input(shape=(None,))
encoder_padding_mask = keras.Input(shape=(None,))
decoder_token_ids = keras.Input(shape=(None,))
decoder_padding_mask = keras.Input(shape=(None,))

token_embedding_layer = keras_nlp.layers.ReversibleEmbedding(vocab_size, hidden_dim)
encoder_token_embedding = token_embedding_layer(encoder_token_ids)
encoder_pos_embedding = keras_nlp.layers.PositionEmbedding(max_seq_len)(encoder_token_embedding)

x = encoder_token_embedding + encoder_pos_embedding
x = layers.LayerNormalization()(x)
x = layers.Dropout(dropout)(x)

for _ in range(num_layers):
    x = transformer_encoder(x, encoder_padding_mask, dropout, activation=activation)
encoder_output = x

decoder_token_embedding = token_embedding_layer(decoder_token_ids)
decoder_pos_embedding = keras_nlp.layers.PositionEmbedding(max_seq_len)(decoder_token_embedding)

x = decoder_token_embedding + decoder_pos_embedding
x = layers.LayerNormalization()(x)
x = layers.Dropout(dropout)(x)

for _ in range(num_layers):
    x = transformer_decoder(x, encoder_output, decoder_padding_mask, encoder_padding_mask,
                            dropout, activation=activation)
decoder_output = token_embedding_layer(x, reverse=True)

model = keras.Model(inputs=(encoder_token_ids, encoder_padding_mask,
                            decoder_token_ids, decoder_padding_mask),
                    outputs=(encoder_output, decoder_output))
model.summary()

2026-04-08 10:11:48.832457: E external/local_xla/xla/stream_executor/cuda/cuda_platform.cc:51] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, None)      │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ reversible_embeddi… │ (None, None,      │ 38,603,520 │ input_layer[0][0… │
│ (ReversibleEmbeddi… │ 50265)            │            │ input_layer_2[0]… │
│                     │                   │            │ layer_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ position_embedding  │ (None, None, 768) │    786,432 │ reversible_embed… │
│ (PositionEmbedding) │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add (Add)           │ (None, None, 768) │          0 │ reversible_embed… │
│                     │                   │            │ position_embeddi… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_normalization │ (None, None, 768) │      1,536 │ add[0][0]         │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ input_layer_1       │ (None, None)      │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout (Dropout)   │ (None, None, 768) │          0 │ layer_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ multi_head_attenti… │ (None, None, 768) │  2,362,368 │ input_layer_1[0]… │
│ (MultiHeadAttentio… │                   │            │ dropout[0][0],    │
│                     │                   │            │ dropout[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_2 (Dropout) │ (None, None, 768) │          0 │ multi_head_atten… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_1 (Add)         │ (None, None, 768) │          0 │ dropout_2[0][0],  │
│                     │                   │            │ dropout[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_normalizatio… │ (None, None, 768) │      1,536 │ add_1[0][0]       │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense (Dense)       │ (None, None,      │  2,362,368 │ layer_normalizat… │
│                     │ 3072)             │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_1 (Dense)     │ (None, None, 768) │  2,360,064 │ dense[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_3 (Dropout) │ (None, None, 768) │          0 │ dense_1[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_2 (Add)         │ (None, None, 768) │          0 │ dropout_3[0][0],  │
│                     │                   │            │ layer_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_normalizatio… │ (None, None, 768) │      1,536 │ add_2[0][0]       │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ multi_head_attenti… │ (None, None, 768) │  2,362,368 │ input_layer_1[0]… │
│ (MultiHeadAttentio… │                   │            │ layer_normalizat… │
│                     │                   │            │ layer_normalizat

 Total params: 139,417,344 (531.83 MB)

 Trainable params: 139,417,344 (531.83 MB)

 Non-trainable params: 0 (0.00 B)

### 사전 훈련된 BART 모델로 텍스트 생성하기

In [7]:
bart_lm = keras_nlp.models.BartSeq2SeqLM.from_preset('bart_base_en')

In [8]:
sampler = keras_nlp.samplers.TopKSampler(k=10, temperature=10, seed=42)
bart_lm.compile(sampler=sampler)
bart_lm.generate('I like coffee because it helps me wake up in the morning.', max_length=20)

2026-04-08 10:12:04.373879: I external/local_xla/xla/service/service.cc:163] XLA service 0x78237803cdd0 initialized for platform Host (this does not guarantee that XLA will be used). Devices:
2026-04-08 10:12:04.373916: I external/local_xla/xla/service/service.cc:171]   StreamExecutor device (0): Host, Default Version
I0000 00:00:1775610724.398272   41936 device_compiler.h:196] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


'ItI like my cup and my tea - coffee that has been good in my brain is'

In [9]:
sampler = keras_nlp.samplers.TopKSampler(k=10, temperature=10, seed=42)
bart_lm.compile(sampler=sampler)
bart_lm.generate(
    {
        'encoder_text': 'I hate coffee, so I always drink tea instead.',
        'decoder_text': 'In the morning, when I wake up'
    },
    max_length=20
)

'In the morning, when I wake up at the same temperature it was already hot; coffee'

### 허깅페이스 BART 모델로 텍스트 요약하기

In [10]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

model_name = "facebook/bart-large-cnn"

# 1. 토크나이저와 모델 직접 불러오기 (포장지 없이 알맹이만 가져옵니다)
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

text = """
The James Webb Space Telescope is a space telescope designed primarily to conduct infrared astronomy. 
As the largest telescope in space, it is equipped with high-resolution and highly sensitive instruments, 
allowing it to view objects too old, distant, or faint for the Hubble Space Telescope.
"""

# 2. 텍스트를 모델이 이해할 수 있는 숫자(텐서)로 변환
inputs = tokenizer(text, max_length=1024, return_tensors="pt", truncation=True)

# 3. 모델을 통해 요약 결과(숫자 배열) 생성
summary_ids = model.generate(
    inputs["input_ids"], 
    max_length=50, 
    min_length=10, 
    num_beams=4, # 더 좋은 품질의 문장을 찾기 위한 옵션
    early_stopping=True
)

# 4. 생성된 숫자를 다시 사람이 읽을 수 있는 텍스트로 변환
result = tokenizer.decode(summary_ids[0], skip_special_tokens=True)

print("요약 결과:", result)

# pipe = pipeline("summarization", model="facebook/bart-large-cnn", framework="pt")

Please make sure the generation config includes `forced_bos_token_id=0`. 
Loading weights: 100%|██████████| 511/511 [00:00<00:00, 1063.11it/s]


요약 결과: The James Webb Space Telescope is a space telescope designed primarily to conduct infrared astronomy. It is equipped with high-resolution and highly sensitive instruments, allowing it to view objects too old, distant, or faint for the Hubble Space Telescope


In [11]:
import torch
from transformers import set_seed

ENG_TEXT = """
Voyager 1 is a space probe launched by NASA on September 5, 1977, as part of the Voyager program to study the outer Solar System and the interstellar space beyond the Sun's heliosphere. It was launched 16 days after its twin, Voyager 2. It communicates through the NASA Deep Space Network (DSN) to receive routine commands and to transmit data to Earth. Real-time distance and velocity data are provided by NASA and JPL. At a distance of 162.7 AU (24.3 billion km; 15.1 billion mi) from Earth as of May 2024, it is the most distant humanmade object from Earth.
"""

# 시드 고정 (동일한 결과를 얻기 위해)
set_seed(42)

# 1. 텍스트를 모델이 이해할 수 있는 숫자(텐서)로 변환
inputs = tokenizer(ENG_TEXT, max_length=1024, return_tensors="pt", truncation=True)

# 2. 모델을 통해 텍스트 생성 (원하시던 파라미터 그대로 적용!)
summary_ids = model.generate(
    inputs["input_ids"], 
    max_length=70, 
    do_sample=True, 
    top_k=10, 
    temperature=3.0  # 창의성(무작위성)을 아주 높게 설정하셨네요!
)

# 3. 생성된 숫자를 다시 사람이 읽을 수 있는 텍스트로 변환
result = tokenizer.decode(summary_ids[0], skip_special_tokens=True)

print("결과:", result)

# ENG_TEXT = """
# Voyager 1 is a space probe launched by NASA on September 5, 1977, as part of the Voyager program to study the outer Solar System and the interstellar space beyond the Sun's heliosphere. It was launched 16 days after its twin, Voyager 2. It communicates through the NASA Deep Space Network (DSN) to receive routine commands and to transmit data to Earth. Real-time distance and velocity data are provided by NASA and JPL. At a distance of 162.7 AU (24.3 billion km; 15.1 billion mi) from Earth as of May 2024, it is the most distant humanmade object from Earth.
# """
# set_seed(42)
# pipe(ENG_TEXT, max_length=70, do_sample=True, top_k=10, temperature=3.0)

결과: Voyager 1 was launched on 5 September 1977. It is the most distant humanmade object from Earth. As of May 2024, Voyager will be 162.7 AU (24.3 billion km; 15.1 billion mi) in distance. Voyager 1 is part of a program to study the outer Solar System and interstellar space


In [12]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import torch

# 한국어 요약에 특화된 KoBART 모델 지정
model_name = "digit82/kobart-summarization"

# 1. 파이프라인 없이 토크나이저와 모델 직접 불러오기
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

# 테스트용 한국어 텍스트 (원하시는 기사나 텍스트로 바꿔보세요!)
ko_text = """
국내 연구진이 상온·상압 초전도체라고 주장한 물질 'LK-99'를 검증해온 한국초전도저온학회 검증위원회가 "LK-99는 초전도체가 아니다"라는 결론을 내리고 활동을 종료했다. 검증위는 13일 배포한 백서에서 "원논문의 데이터와 국내외 재현실험 연구결과를 종합해 고려한 결과, LK-99가 상온·상압 초전도체라는 근거는 전혀 없다"고 밝혔다.
"""

# 2. 텍스트를 숫자로 변환
inputs = tokenizer(ko_text, max_length=1024, return_tensors="pt", truncation=True)

# 3. 모델을 통해 요약 결과 생성
summary_ids = model.generate(
    inputs["input_ids"], 
    max_length=128,       # 요약문의 최대 길이
    min_length=10,        # 요약문의 최소 길이
    num_beams=4,          # 문장 품질을 높이기 위한 빔 서치
    eos_token_id=tokenizer.eos_token_id # 문장이 끝났음을 알려주는 토큰
)

# 4. 생성된 숫자를 다시 텍스트로 변환하여 출력
result = tokenizer.decode(summary_ids[0], skip_special_tokens=True)

print("📝 원본 길이:", len(ko_text))
print("✨ 요약 결과:", result)
# kobart_pipe = pipeline("text2text-generation", model="digit82/kobart-summarization")

Loading weights: 100%|██████████| 262/262 [00:00<00:00, 7456.39it/s]


📝 원본 길이: 192
✨ 요약 결과: 한국초전도저온학회 검증위원회가 13일 배포한 백서에서 "원논문의 데이터와 국내외 재현실험 연구결과를 종합해 고려한 결과, LK-99가 초전도체라는 근거는 전혀 없다"고 밝혔다.


In [13]:
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, set_seed

# 혹시 메모리에서 날아갔을까 봐 모델과 토크나이저를 다시 안전하게 불러옵니다.
model_name = "digit82/kobart-summarization"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

KOR_TEXT = """
2023-2024년 쉰드흐누퀴르 분화는 2023년 12월 18일 저녁 아이슬란드 그린다비크에 있는 쉰드흐누퀴르 분화구에서 화산 폭발이 발생해 지상에 있는 열극에서 용암이 분출한 사건이다. 용암 분출과 뒤따른 지진 활동 빈도는 다음 날인 2023년 12월 19일부터 감소했으나 새로 열린 열극의 양쪽에서 용암이 옆으로 넓게 퍼져나갔다. 이번 분화는 2021년 분화 시작 이래 쉬뒤르네스에서 일어난 가장 큰 분화로 최대 100 m 높이의 용암 분수가 관측되었으며 분화지에서 약 42 km 떨어진 아이슬란드의 수도 레이캬비크에서도 화산 분화 장면을 볼 수 있었다. 화산 분화는 2023년 12월 21일 화산 상공 관측 결과 더 이상의 용암 분출이 보이지 않아 종료되었으나 아이슬란드 기상청은 "분화 종식을 선언하기에는 너무 이르다"며 지속적으로 관측하겠다고 말했다. 쉰드흐누퀴르는 현재 화산지대이자 쉬뒤르네스 열곡대의 활성 열극에 속한다.
"""

# 시드 고정
set_seed(42)

# 1. 텍스트를 모델이 이해하는 숫자로 변환
inputs = tokenizer(KOR_TEXT, return_tensors="pt", max_length=1024, truncation=True)

# 2. 캡처해주신 옵션 그대로 모델에 넣고 요약 결과 생성!
summary_ids = model.generate(
    inputs["input_ids"], 
    max_length=100, 
    do_sample=True, 
    top_k=10, 
    temperature=2.0, 
    eos_token_id=1
)

# 3. 사람이 읽을 수 있는 텍스트로 다시 변환
result = tokenizer.decode(summary_ids[0], skip_special_tokens=True)

print("✨ 결과:", result)

# KOR_TEXT = """
# 2023-2024년 쉰드흐누퀴르 분화는 2023년 12월 18일 저녁 아이슬란드 그린다비크에 있는 쉰드흐누퀴르 분화구에서 화산 폭발이 발생해 지상에 있는 열극에서 용암이 분출한 사건이다. 용암 분출과 뒤따른 지진 활동 빈도는 다음 날인 2023년 12월 19일부터 감소했으나 새로 열린 열극의 양쪽에서 용암이 옆으로 넓게 퍼져나갔다. 이번 분화는 2021년 분화 시작 이래 쉬뒤르네스에서 일어난 가장 큰 분화로 최대 100 m 높이의 용암 분수가 관측되었으며 분화지에서 약 42 km 떨어진 아이슬란드의 수도 레이캬비크에서도 화산 분화 장면을 볼 수 있었다. 화산 분화는 2023년 12월 21일 화산 상공 관측 결과 더 이상의 용암 분출이 보이지 않아 종료되었으나 아이슬란드 기상청은 "분화 종식을 선언하기에는 너무 이르다"며 지속적으로 관측하겠다고 말했다. 쉰드흐누퀴르는 현재 화산지대이자 쉬뒤르네스 열곡대의 활성 열극에 속한다.
# """
# set_seed(42)
# kobart_pipe(KOR_TEXT, max_length=100, do_sample=True,
#             top_k=10, temperature=2.0, eos_token_id=1)

Loading weights: 100%|██████████| 262/262 [00:00<00:00, 6406.91it/s]


✨ 결과:  2011년드흐누퀴르에 위치한 쉰드흐흐누퀴르 분화구에서는 화산 분출 사건이 있었다.
